# Write Up

In the current set up for generating scalograms, the event for event labeling is per question. Each question is stored as a csv under a participant directory. Thus, for the event tagging, it is done by creating that per question csv, and therefore uses a question based labeling approach. Event labeling is applied in the time domain by assigning each EEG segment with its question ID for every participant before the actual generation of scalograms occurs.

In addition, this notebook demonstrates loading in the data, segmenting the data, and generating grey-scaled scalograms in the form of numpy arrays (stored in my google drive)

# Set Up

In [ ]:
import os, re, glob
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import drive

import pywt
from scipy.signal import detrend
from PIL import Image
import matplotlib.cm as cm
import matplotlib


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


## Getting data

In [ ]:
# ZIP_DIR="/content/drive/MyDrive/EEG_Data"
# ZIP_NAME="Cal-Poly-EEG-Data-main.zip"
# OUT_DIR="/content/drive/MyDrive/EEG_Data"

# !mkdir -p "$OUT_DIR"
# !unzip -q -o "$ZIP_DIR/$ZIP_NAME" -d "$OUT_DIR"
# !ls -lah "$OUT_DIR"


I am only looking at questions 1 - 40

In [ ]:
# NOTE: Code adopted from Hannah Moshtaghi

# --- Configuration ---
data_root = '/content/drive/MyDrive/EEG_Data/Cal-Poly-EEG-Data-main/raw/2'         # Change to 'OrganizedRSCA/2' for study 2
output_root = 'MergedRSCA'            # Output directory for merged data
MERGE_ALL = False                      # <<< Toggle: True for full merge only, False for group-by-group

# --- Define question groups by index range ---
all_qs = range(1,40)

question_groups = {
  f"Question{q:02d}": [q] for q in all_qs
}

# --- Main Loop: Process Each Study Folder ---
for study_folder in os.listdir(data_root):
    study_path = os.path.join(data_root, study_folder)
    out_study_path = os.path.join(output_root, study_folder)

    if not os.path.isdir(study_path): # Added check here
        continue

    for person_folder in os.listdir(study_path):
        person_path = os.path.join(study_path, person_folder)

        if not os.path.isdir(person_path):
            continue

        csv_files = sorted(glob.glob(os.path.join(person_path, '*.csv')))
        file_map = {}

        for file in csv_files:
            parts = os.path.basename(file).split('_')
            if len(parts) >= 4:
                try:
                    seq_num = int(parts[2])  # Assumes format like: PREFIX_<ID>_<QNUM>_SUFFIX.csv
                    file_map[seq_num] = file
                except ValueError:
                    continue

        person_id = os.path.basename(person_path)
        merged_out_dir = os.path.join(out_study_path, person_id)
        os.makedirs(merged_out_dir, exist_ok=True)

        if MERGE_ALL:
            # --- Merge all files into one, skip group merging ---
            all_dataframes = []
            # Sort file_map by key (sequence number) before processing
            for seq_num in sorted(file_map.keys()):
                file = file_map[seq_num]
                df = pd.read_csv(file)
                all_dataframes.append(df)

            if all_dataframes:
                full_merged = pd.concat(all_dataframes, ignore_index=True)
                full_merged.dropna(inplace=True)

                full_merged_path = os.path.join(merged_out_dir, f"{person_id}_all.csv")
                full_merged.to_csv(full_merged_path, index=False)
                print(f"Saved merged all file: {full_merged_path}")
        else:
            # --- Group-by-group merge ---
            all_missing = {}

            for group_name, questions in question_groups.items():
                missing = []
                dataframes = []

                # Sort question numbers before processing
                for q_num in sorted(questions):
                    if q_num not in file_map:
                        missing.append(q_num)
                    else:
                        df = pd.read_csv(file_map[q_num])
                        dataframes.append(df)

                if missing:
                    all_missing[group_name] = missing
                    print(f"Skipped {group_name} for {person_id} due to missing files: {missing}")
                    continue

                if dataframes:
                    merged_df = pd.concat(dataframes, ignore_index=True)
                    merged_df.dropna(inplace=True)

                    merged_path = os.path.join(merged_out_dir, f"{person_id}_{group_name}.csv")
                    merged_df.to_csv(merged_path, index=False)
                    print(f"Saved merged file: {merged_path}")

            # --- Save missing summary ---
            if all_missing:
                missing_path = os.path.join(merged_out_dir, f"{person_id}_missing_questions.txt")
                with open(missing_path, 'w') as f:
                    for group, missing_list in all_missing.items():
                        f.write(f"{group}: missing question numbers {missing_list}\n")


Skipped Question01 for 67c2836837717e8573ee0670 due to missing files: [1]
Skipped Question02 for 67c2836837717e8573ee0670 due to missing files: [2]
Skipped Question03 for 67c2836837717e8573ee0670 due to missing files: [3]
Skipped Question04 for 67c2836837717e8573ee0670 due to missing files: [4]
Skipped Question05 for 67c2836837717e8573ee0670 due to missing files: [5]
Skipped Question06 for 67c2836837717e8573ee0670 due to missing files: [6]
Skipped Question07 for 67c2836837717e8573ee0670 due to missing files: [7]
Skipped Question08 for 67c2836837717e8573ee0670 due to missing files: [8]
Skipped Question09 for 67c2836837717e8573ee0670 due to missing files: [9]
Skipped Question10 for 67c2836837717e8573ee0670 due to missing files: [10]
Skipped Question11 for 67c2836837717e8573ee0670 due to missing files: [11]
Skipped Question12 for 67c2836837717e8573ee0670 due to missing files: [12]
Skipped Question13 for 67c2836837717e8573ee0670 due to missing files: [13]
Skipped Question14 for 67c28368377

## Helper Functions


In [ ]:
# This function basically takes in the eeg signal, the specified window_size and stride and returns an
# array that has the various windows
def window_signal(sig, window_size, stride):
    starts = np.arange(0, max(len(sig) - window_size + 1, 0), stride)
    return np.stack([sig[s:s+window_size] for s in starts], axis=0) if len(starts) else np.empty((0, window_size))
    # Just a total windows by window size array of eeg values (n_windows, window_size)

def cwt_tile(signal_1d, fs, fmin=4, fmax=40, num_freqs=64, log_power=True):
    signal_1d = detrend(signal_1d, type="linear") # gets rid of linear drift (not sure if helpful especially since its per window)

    target_freqs = np.geomspace(fmin, fmax, num_freqs) # num_freqs from fmin to fmax. E.G. 64 freqs between 4 to 40 Hz
    central = pywt.central_frequency("morl")
    scales = central * fs / target_freqs # maps freq to CWT scales for Morlet

    coeffs, freqs = pywt.cwt(signal_1d, scales=scales, wavelet="morl", sampling_period=1/fs)
    power = (coeffs.real**2 + coeffs.imag**2) if np.iscomplexobj(coeffs) else np.abs(coeffs)**2

    # for when we want log_power for easier viewing
    if log_power:
        power = np.log10(power + 1e-12)

    # just filters outliers so the graphs are not dominated by outliers
    lo, hi = np.percentile(power, [5, 99])
    tile = np.clip((power - lo) / (hi - lo + 1e-12), 0, 1)

    return tile.astype(np.float32), freqs
    # returns scaolograms as tiles power x freq x time

# resizes images to 64, 64 -- mostly for if num_freq is changed to smth other than 64, not sure if resizing is the move here tho, will test later
def resize_tile(tile, out_size=(64, 64)):
    # PIL expects HxW, so treat F as H and T as W
    arr = (tile * 255.0).astype(np.uint8)
    #im = Image.fromarray(arr, mode="L")
    im = Image.fromarray(arr.astype(np.uint8))
    im = im.resize(out_size, Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)

# # saves a colored png
# def save_tile_png(tile_float, out_path, cmap_name = "viridis"):
#     os.makedirs(os.path.dirname(out_path), exist_ok=True)
#     #cmap = cm.get_cmap(cmap_name)
#     cmap = matplotlib.colormaps.get_cmap(cmap_name)
#     colored = (cmap(tile_float)[..., :3] * 255).astype(np.uint8)
#     #Image.fromarray(colored, mode="RGB").save(out_path, optimize=True)
#     Image.fromarray(colored).convert("RGB").save(out_path, optimize=True)


# ----------------------------
# HELPERS RELATED TO: these ones for dealing with specific structure of how data loaded in
# ----------------------------

# Yield participant directories (immediate children that are directories).
def iter_participants(group_dir: Path):
    for p in sorted(group_dir.iterdir()):
        if p.is_dir():
            yield p

# recursively returns the csv under the participant folder
def list_question_csvs(person_dir: Path):
    return sorted(person_dir.rglob("*.csv"))

# Given person, return the corresponding person, question, channel pathing
def output_dir_for(group: str, participant_id: str, q_num: int, channel: str) -> Path:
    channel_folder = channel.replace(" ", "_")
    return OUT_ROOT / group / participant_id / f"Question{q_num:02d}" / channel_folder

# Saves each tile as winXXXXX.npy with skipping any file that already exists
def save_tiles_resume(tiles, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    for i, tile in enumerate(tiles):
        out_path = out_dir / f"win{i:05d}.npy"
        if out_path.exists():
            continue  # this adds in resume behavior

        tile_f32 = tile.astype(np.float32) / 255.0
        np.save(out_path, tile_f32)

# Add later
def process_csv(csv_path: Path, group: str, participant_id: str):
    base = csv_path.name
    mat = csv_name_format.match(base)
    if not mat:
        # Not a Question CSV, thus ignore
        return 0

    q_num = int(mat.group(1))
    df = pd.read_csv(csv_path)

    n_saved_total = 0

    for channel in channel_names:
        if channel not in df.columns:
            # If a CSV is missing a channel, skip that channel only.
            print(f"[WARN] Missing '{channel}' in {csv_path}")
            continue

        sig = np.asarray(df[channel], dtype=float).reshape(-1)
        wins = window_signal(sig, WINDOW_SIZE, STRIDE)

        # Make scalogram tiles
        tiles = []
        for w in wins:
            tile, freqs = cwt_tile(
                w, fs,
                fmin=4, fmax=40,
                num_freqs=64,
                log_power=True
            )
            tile_img = resize_tile(tile, out_size=(64, 64))
            tiles.append(tile_img)

        out_dir = output_dir_for(group, participant_id, q_num, channel)
        before = len(list(out_dir.glob("win*.npy"))) if out_dir.exists() else 0

        save_tiles_resume(tiles, out_dir)

        after = len(list(out_dir.glob("win*.npy")))
        n_saved = max(0, after - before)
        n_saved_total += n_saved

    return n_saved_total

# Generating/Saving Scalograms

In [ ]:
# Pathing stuff
DATA_ROOT = Path("/content/MergedRSCA")

OUT_ROOT = Path("/content/drive/MyDrive/Scalograms_numpy")
OUT_ROOT.mkdir(parents=True, exist_ok=True)


# Specifying inputs; defining needed variables
GROUPS = ["control", "fixed", "growth"]

# ON CHANNEL N (we just got the 1-4)
fs = 256
WINDOW_SIZE = 64 #idk why 64, just standard
STRIDE = 32 # 50% overlap so half of whats in cur is old

channel_names = ["Channel 1", "Channel 2", "Channel 3", "Channel 4"]

csv_name_format = re.compile(r".*_Question(\d+)\.csv$", re.IGNORECASE)

total_saved = 0
total_csvs = 0


In [ ]:
# Actually creating the scalograms and saving it
for group in GROUPS:
    group_dir = DATA_ROOT / group
    if not group_dir.exists():
        print(f"[WARNING] Group folder missing: {group_dir} ( skipping :( )")
        continue

    print(f"\n=== GROUP: {group} ===")

    for person_dir in iter_participants(group_dir):
        participant_id = person_dir.name
        csv_paths = list_question_csvs(person_dir)

        print(f"\nParticipant {participant_id}: found {len(csv_paths)} CSV files")

        for csv_path in csv_paths:
            # Only Question CSVs will be processed; others ignored
            saved_now = process_csv(csv_path, group, participant_id)
            if saved_now is not None:
                total_saved += saved_now
            total_csvs += 1

print("\nDONE :o")
print(f"Total CSV files scanned: {total_csvs}")
print(f"Total new images saved (approx): {total_saved}")
print(f"Output root: {OUT_ROOT}")


=== GROUP: control ===

Participant 67c26b6a6aed7a8612189b14: found 32 CSV files

Participant 67c26fde6aed7a861218a3cb: found 32 CSV files

Participant 67c27401f112be75fec84544: found 32 CSV files

Participant 67c28656c6360e2457d02f60: found 32 CSV files

Participant 67c660bb89c7d2ddeb14ebf0: found 32 CSV files

Participant 67ca1c5c8a63c0deed06b10a: found 0 CSV files

Participant 67ca223b8ec09603053684b8: found 32 CSV files

Participant 67ca26ff8ec0960305368cd7: found 32 CSV files

Participant 67d0a48d0b0cf28d843388f3: found 32 CSV files

Participant 68083b0c332567ffa5217126: found 32 CSV files

Participant 681bc78d7f0a181a7e96a598: found 32 CSV files

=== GROUP: fixed ===

Participant 67c269abbbbc97c24e64d8dd: found 0 CSV files

Participant 67c65ae75af132787b58df9a: found 32 CSV files

Participant 67c65b5db393e8e43f8a87bc: found 32 CSV files

Participant 67c65d88bc1477587c80aaa7: found 0 CSV files

Participant 67c77911ff18ef1bd5d4ca8a: found 32 CSV files

Participant 67c77c008cb8fc57